# v9c CrossJEPA — Method 1 (single, dual_heads, or teacher_id)

Trains the 3D-volume → 2D-slice CrossJEPA tower. Three teacher-architecture choices:

| `--mode`        | Teachers | Behavior |
|---|---|---|
| `single`        | one     | v1 setup, single teacher (v8 or dinov2) — backwards-compat |
| `dual_heads`    | 2+      | **Option A.** Predictor has one output head per teacher; loss = weighted sum |
| `teacher_id`    | 2+      | **Option C.** Single output head; one teacher sampled per step, teacher_id fed as gradient sink |

**Recommended**: `--mode dual_heads --teachers v8,dinov2` (combines v8's medical tumor signal with DINOv2's intensity-robust general features — fix for the preprocessing-bias inversion caught on IXI held-out).

**Uploads needed (one-time):**
1. `MyDrive/colab_bundle.zip` — source code
2. `MyDrive/crossjepa_data_v2.zip` — 17 GB multi-pipeline data (radiata + IXI + BraTS)

v8 teacher .pt and DINOv2 base are pulled at runtime from HF Models / HuggingFace Hub.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -10

## 2. Install dependencies

In [ ]:
%pip install -q nibabel==5.* huggingface_hub>=0.20 segmentation-models-pytorch>=0.3.3 timm>=0.9.16

## 3. Unzip the source bundle (from Drive)

Upload `colab_bundle.zip` (built locally by `scripts/rebundle_colab.py`) to `MyDrive/` once. This cell unzips it to `/content/neurolens/`.

In [ ]:
import os, sys
BUNDLE = '/content/drive/MyDrive/colab_bundle.zip'
DEST = '/content/neurolens'
!rm -rf {DEST}
!mkdir -p {DEST}
!unzip -q -o {BUNDLE} -d {DEST}
sys.path.insert(0, DEST)
os.chdir(DEST)
!ls src/research/v9c_crossjepa/
!ls src/train_v9c_*

## 4. Unzip the dataset bundle (from Drive)

Upload `crossjepa_data.zip` (built locally by `scripts/build_crossjepa_dataset_bundle.py`, ~13 GB) to `MyDrive/` once. This cell unzips both `healthy/` (800 radiata-ai T1) and `brats/` (1251 BraTS-2021 4-modality) to `/content/data/`.

In [ ]:
DATA_BUNDLE = '/content/drive/MyDrive/crossjepa_data_v2.zip'
DATA_DEST = '/content/data'
!rm -rf {DATA_DEST}
!mkdir -p {DATA_DEST}
!unzip -q -o {DATA_BUNDLE} -d {DATA_DEST}
import subprocess
n_rad = subprocess.run(['bash', '-c', f'find {DATA_DEST}/healthy/radiata -name "*.nii.gz" | wc -l'], capture_output=True, text=True).stdout.strip()
n_ixi = subprocess.run(['bash', '-c', f'find {DATA_DEST}/healthy/ixi -name "*.nii.gz" | wc -l'], capture_output=True, text=True).stdout.strip()
n_brats = subprocess.run(['bash', '-c', f'find {DATA_DEST}/brats -name "*.nii.gz" | wc -l'], capture_output=True, text=True).stdout.strip()
print(f'unzipped: {n_rad} radiata + {n_ixi} ixi healthy + {n_brats} BraTS NIfTI')

## 5. Pull the frozen v8 UNet checkpoint (one-time, ~383 MB)

This is the **only** network download in this notebook. The v8 PyTorch state dict lives in our public HF Models repo.

In [ ]:
from huggingface_hub import hf_hub_download
V8_CKPT = hf_hub_download(
    repo_id='Tubai01/neurolens-models', repo_type='model',
    filename='attention_unet_v8/best_micro.pt',
)
print(f'v8 ckpt at: {V8_CKPT}')
!ls -lh {V8_CKPT}

## 6. Smoke-test the build

Confirms the 3D ViT + predictor + frozen v8 teacher construct, a single forward+backward step works on this GPU, and the frozen-teacher invariant holds.

In [ ]:
import glob, torch, time
from torch.utils.data import DataLoader
from src.research.v9c_crossjepa.dataset_3d import Vol2SliceDataset
from src.research.v9c_crossjepa.v8_teacher import V8FrozenTeacher
from src.research.v9c_crossjepa.volume_to_slice import Vol2SliceModel

device = 'cuda'
torch.manual_seed(0)

# Load v8 frozen teacher
print('[smoke] loading v8 teacher...')
teacher = V8FrozenTeacher.from_unet_checkpoint(
    V8_CKPT,
    encoder_name='tu-convnext_tiny.fb_in22k_ft_in1k',
    in_channels=3, image_size=384, device=device,
)
snapshot = [p.clone() for p in teacher.encoder.parameters()]

# Build model
model = Vol2SliceModel(
    v8_teacher=teacher,
    volume_size=(144, 192, 192), in_chans=1, patch_size=16,
    encoder_dim=384, encoder_depth=12, encoder_heads=6,
    predictor_dim=192, predictor_depth=6,
).to(device)
print(f'[smoke] trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f} M')

# Single-step forward+backward on a tiny dataset slice
scans = sorted(glob.glob('/content/data/healthy/*/sub-*/ses-*/anat/*.nii.gz'))[:4]
print(f'[smoke] using {len(scans)} healthy scans for the smoke step')
ds = Vol2SliceDataset(scan_paths=scans, volume_size=(144, 192, 192),
                       in_channels=1, slices_per_volume=1)
def _coll(b):
    keys = ('volume','target_slice_rgb','plane_idx','slice_idx_norm','voxel_spacing','intensity_hist')
    return {k: torch.stack([s[k] for s in b]) for k in keys}
loader = DataLoader(ds, batch_size=1, collate_fn=_coll)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
for batch in loader:
    for k, v in batch.items():
        if torch.is_tensor(v): batch[k] = v.to(device, non_blocking=True)
    t0 = time.perf_counter()
    with torch.amp.autocast('cuda'):
        out = model.training_step(batch)
    out['loss'].backward(); opt.step(); opt.zero_grad()
    print(f'[smoke] step OK in {time.perf_counter()-t0:.2f}s  loss={float(out["loss"]):.4f}  cos_sim={float(out["cos_sim"]):.4f}')
    print(f'        gpu_mem = {torch.cuda.memory_allocated()/1e9:.2f} GB')
    break

# Verify teacher untouched
unchanged = all(torch.equal(s, p) for s, p in zip(snapshot, teacher.encoder.parameters()))
print(f'[smoke] frozen-teacher invariant: {"PASS" if unchanged else "FAIL — teacher params changed!"}')

## 7. Train Method 1 end-to-end

Saves to Drive (`MyDrive/v9c_crossjepa_method1/last.pt`) every 200 steps + end of epoch, so a Colab disconnect doesn't lose progress. Resumes automatically via `--resume auto`.

In [ ]:
# OPTION A (recommended): dual-teacher v8 + DINOv2
# Each teacher has its own output head; loss = smooth_L1(pred_v8, v8_target) + λ*smooth_L1(pred_dinov2, dinov2_target)
MODE = 'dual_heads'        # or 'teacher_id' for Option C, or 'single' for the v1 setup
TEACHERS = 'v8,dinov2'      # comma-separated; for --mode single use just one (e.g. 'dinov2')
WEIGHTS = '1.0,1.0'         # ignored in --mode teacher_id
OUTPUT_DIR = '/content/drive/MyDrive/v9c_crossjepa_method1_dualheads'

!python src/train_v9c_method1_vol2slice.py --scans_glob '/content/data/healthy/**/*.nii.gz' --v8_ckpt {V8_CKPT} --mode {MODE} --teachers {TEACHERS} --teacher_weights {WEIGHTS} --output_dir {OUTPUT_DIR} --volume_size 144 192 192 --in_channels 1 --patch_size 16 --encoder_dim 384 --encoder_depth 12 --encoder_heads 6 --predictor_dim 192 --predictor_depth 6 --batch_size 2 --slices_per_volume 6 --epochs 30 --lr 2e-4 --num_workers 2 --amp --checkpoint_every_steps 200 --resume auto --augment --aug_intensity_jitter 0.20 --aug_contrast_jitter 0.30 --aug_gamma_jitter 0.30 --aug_bias_field_strength 0.15 --aug_noise_std 0.02 --aug_renormalize_pct 0.5

## 3-experiment teacher ablation — run each in its own session

After v2 with v8 teacher showed raw AUC = 0.00 inverted (preprocessing-bias
intact, z-score recovers AUC 0.999), the empirical question is: does a
different teacher fix the raw signal? Three candidates, each is a one-line
edit to the training cell vars above. Same data, same epochs, same evaluator.

**(1) DINOv2 only** — best LOSO evidence (AUC 0.842), intensity-robust by training
```python
MODE = 'single'
TEACHERS = 'dinov2'
OUTPUT_DIR = '/content/drive/MyDrive/v9c_crossjepa_method1_dinov2'
```

**(2) MedSAM only** — medical-domain teacher, 1.5M training pairs (much broader than v8's BraTS-only)
```python
MODE = 'single'
TEACHERS = 'medsam'
OUTPUT_DIR = '/content/drive/MyDrive/v9c_crossjepa_method1_medsam'
# First-run cost: ~358 MB MedSAM weights auto-download from HF Hub (~30s)
```

**(3) Dual: DINOv2 + MedSAM (Option A)** — combine intensity-robust + medical-domain
```python
MODE = 'dual_heads'
TEACHERS = 'dinov2,medsam'
WEIGHTS = '1.0,1.0'
OUTPUT_DIR = '/content/drive/MyDrive/v9c_crossjepa_method1_dual_dinov2_medsam'
```

## After each training, eval locally

```bash
# raw AUC ≥ 0.85 = teacher fixed the preprocessing bias
python scripts/eval_method1_heldout_ixi.py
# z-score is the polish/fallback; should be ≥ 0.95 for any sensible teacher
python scripts/eval_method1_zscore_heldout.py
```

## Decision table

| Experiment | raw AUC ≥ 0.85? | Interpretation |
|---|---|---|
| dinov2 only | yes | Ship DINOv2 single-teacher, retire v8 from teaching role |
| dinov2 only | no | DINOv2 doesn't fix it either — preprocessing-bias is the model's, not the teacher's |
| medsam only | yes & better than dinov2 | Ship MedSAM single-teacher |
| dual | best of the three | Ship dual — pay 15% compute for combined signal |
| all three give similar z-score AUC | — | Ship v2+z-score as final; revisit only if production needs raw single-slice scoring |

## 8. Method 1c — true cross-modal CrossJEPA (deep bridge)

Method 1 (section 7) had two architectural correctness issues:

| Issue | Why it broke the bridge |
|---|---|
| `intensity_hist` of target slice was fed into the predictor's query | The predictor saw a fingerprint of the answer; gradients didn't have to flow through the 3D encoder to fit the loss. |
| Source = 3D volume, target = 2D slice of the **same** volume | The target is a literal voxel-subset of the input — no semantic gap to bridge. |

**Method 1c fixes both AND deepens the bridge to be genuinely cross-modal:**

1. **`Method1CrossModalConditioning`** drops the intensity histogram. Conditioning carries only `(plane, slice_idx, voxel_spacing, source_modality_id, target_modality_id)` — position + identity, never content.

2. **`Vol2SliceCrossModalDataset`** with three simultaneous differences between source and target:
   - **Dimensionality:** 3D volume → 2D slice
   - **Intensity space:** modality A → modality B (BraTS 4-modality co-registered)
   - **Orientation:** target plane sampled uniformly from {axial, sagittal, coronal} — encoder cannot pixel-map across orientations, so it must build a real 3D anatomical representation

3. **Pair blacklist + weights** emphasize the genuinely hard mappings:
   - `T1↔T1c` is **blacklisted** — near-identity in healthy tissue (gadolinium has nothing to enhance once tumor is filtered out)
   - `T1↔FLAIR` and `T1c↔FLAIR` get weight **3.0** — require tissue classification because FLAIR uses inversion recovery to suppress CSF *specifically*. The encoder cannot just learn pixel-wise remapping; it must know "this voxel is CSF" to predict the suppression.
   - `T1↔T2` and `T1c↔T2` get weight **2.0** — contrast inversion (white matter bright→dark) plus tissue understanding
   - `T2↔FLAIR` gets weight **1.0** — easier (only CSF channel differs)

4. **Tumor-free target filter** (per-plane via the seg mask) keeps the normative pretraining manifold clean — at inference, tumor slices will be out-of-distribution → anomaly signal.

These three orthogonal differences (dimension + orientation + modality) put this in the same structural family as the original CrossJEPA paper's 3D-pointcloud → 2D-image task: the encoder must abstract from the source's intensity statistics AND from the 3D voxel grid toward a tissue-identity-at-spatial-location representation that supports decoding into an arbitrary plane in an arbitrary target modality.

Data: BraTS 4-modality patients only (each has T1+T1c+T2+FLAIR co-registered + seg.nii.gz). The existing `crossjepa_data_v2.zip` already contains all of these via `rglob('*.nii.gz')` — no data-bundle rebuild needed.

In [ ]:
# Method 1c training cell — real cross-modal split (deep bridge)
TEACHER = 'dinov2'          # single teacher; dinov2 is intensity-robust → best fit for cross-modal targets
OUTPUT_DIR_CM = '/content/drive/MyDrive/v9c_crossjepa_method1c_crossmodal'
BRATS_ROOT = '/content/data/brats'   # 4-modality BraTS dir (from crossjepa_data_v2.zip)

# Deep-bridge knobs — defaults are baked into the trainer; passing them
# here makes the design intent explicit in the log.
PAIR_BLACKLIST = 'T1,T1c;T1c,T1'    # T1<->T1c is near-identity in healthy tissue
PAIR_WEIGHTS = ('T1,FLAIR,3;FLAIR,T1,3;T1c,FLAIR,3;FLAIR,T1c,3;'    # tissue classification
                'T1,T2,2;T2,T1,2;T1c,T2,2;T2,T1c,2;'                # contrast inversion
                'T2,FLAIR,1;FLAIR,T2,1')                            # only CSF differs
TARGET_PLANES = 'axial,sagittal,coronal'   # multi-plane forces 3D understanding

!python src/train_v9c_method1_vol2slice.py \
    --mode cross_modal \
    --teachers {TEACHER} \
    --v8_ckpt {V8_CKPT} \
    --brats_root {BRATS_ROOT} \
    --output_dir {OUTPUT_DIR_CM} \
    --pair_blacklist "{PAIR_BLACKLIST}" \
    --pair_weights "{PAIR_WEIGHTS}" \
    --target_planes "{TARGET_PLANES}" \
    --volume_size 144 192 192 --in_channels 1 --patch_size 16 \
    --encoder_dim 384 --encoder_depth 12 --encoder_heads 6 \
    --predictor_dim 192 --predictor_depth 6 \
    --batch_size 2 --pairs_per_volume 6 --epochs 30 --lr 2e-4 \
    --num_workers 2 --amp --checkpoint_every_steps 200 --resume auto \
    --augment --aug_intensity_jitter 0.20 --aug_contrast_jitter 0.30 \
    --aug_gamma_jitter 0.30 --aug_bias_field_strength 0.15 \
    --aug_noise_std 0.02 --aug_renormalize_pct 0.5